# ARIA v2.0 — Terrain Intelligence Upgrade
## 花蓮縣避難所地形風險評估

**延伸 Week 3 ARIA v1.0**：加入 DEM 坡度 + 高程風險分析

**資料來源**：
- 河川中心線：OSM Overpass API
- 避難收容處所：消防署開放資料
- 鄉鎮市區界：國土測繪中心 TGOS
- DEM：內政部 20m DEM（花蓮縣）

**新增分析**：
- 每個避難所 500m 範圍內的平均高程、最大坡度
- 坡度風險分級：LOW (0-15°) / MEDIUM (15-30°) / HIGH (>30°)
- 複合風險 = 河流風險 + 地形風險

## Step 0: 環境設定

In [ ]:
import geopandas as gpd
import pandas as pd
import numpy as np
import rioxarray
import rasterio
from rasterio.mask import mask as rio_mask
from shapely.geometry import mapping
from shapely.ops import unary_union
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from matplotlib.patches import Patch
import json
import warnings
import glob
import tempfile, os

warnings.filterwarnings('ignore')
plt.rcParams['font.family'] = ['Arial Unicode MS', 'Heiti TC', 'sans-serif']

BUFFER_HIGH, BUFFER_MED, BUFFER_LOW = 500, 1000, 2000
DEM_PATH = 'data/dem_20m_hualien.tif'
TARGET_COUNTY = '花蓮縣'
print('Setup complete')

## Step 1: 資料載入與清理

In [ ]:
rivers = gpd.read_file('data/rivers.gpkg').to_crs(epsg=3826)
print(f'河川中心線: {len(rivers)} 條')

shelters_raw = pd.read_csv('data/shelter.csv', encoding='utf-8-sig')
shelters_raw['經度'] = pd.to_numeric(shelters_raw['經度'], errors='coerce')
shelters_raw['緯度'] = pd.to_numeric(shelters_raw['緯度'], errors='coerce')
shelters_raw['預計收容人數'] = pd.to_numeric(shelters_raw['預計收容人數'], errors='coerce').fillna(0).astype(int)
mask = (shelters_raw['經度'].notna() & shelters_raw['緯度'].notna()
    & (shelters_raw['經度'] != 0) & (shelters_raw['緯度'] != 0)
    & shelters_raw['經度'].between(119, 122) & shelters_raw['緯度'].between(21, 26))
shelters = gpd.GeoDataFrame(shelters_raw[mask].reset_index(drop=True),
    geometry=gpd.points_from_xy(shelters_raw[mask]['經度'], shelters_raw[mask]['緯度']),
    crs='EPSG:4326').to_crs(epsg=3826)
print(f'避難所: {len(shelters)} 筆')

shp_files = glob.glob('data/township/TOWN_MOI_*.shp')
townships = gpd.read_file(shp_files[0]).to_crs(epsg=3826)
townships['FULL_NAME'] = townships['COUNTYNAME'] + townships['TOWNNAME']
print(f'鄉鎮市區界: {len(townships)} 區')

## Step 2: 篩選花蓮縣

In [ ]:
hualien_twn = townships[townships['COUNTYNAME'] == TARGET_COUNTY].copy()
hualien_boundary = hualien_twn.dissolve()
shelters_h = gpd.sjoin(shelters, hualien_twn[['geometry', 'FULL_NAME', 'TOWNNAME']],
                        how='inner', predicate='within').drop(columns=['index_right'], errors='ignore')
print(f'花蓮縣鄉鎮: {len(hualien_twn)}, 避難所: {len(shelters_h)}')

## Step 3: 河流緩衝區風險（Week 3 延伸）
三級緩衝區：高風險 ≤500m / 中風險 ≤1000m / 低風險 ≤2000m

In [ ]:
rivers['geometry'] = rivers.geometry.simplify(10)
buf_high_geom = unary_union(rivers.geometry.buffer(BUFFER_HIGH))
buf_med_geom = unary_union(rivers.geometry.buffer(BUFFER_MED))
buf_low_geom = unary_union(rivers.geometry.buffer(BUFFER_LOW))
buf_high = gpd.GeoDataFrame({'rl': ['高']}, geometry=[buf_high_geom], crs=rivers.crs)
buf_med = gpd.GeoDataFrame({'rl': ['中']}, geometry=[buf_med_geom], crs=rivers.crs)
buf_low = gpd.GeoDataFrame({'rl': ['低']}, geometry=[buf_low_geom], crs=rivers.crs)

shelters_h['river_risk'] = '安全'
for buf, label in [(buf_low, '低風險'), (buf_med, '中風險'), (buf_high, '高風險')]:
    idx = gpd.sjoin(shelters_h, buf, how='inner', predicate='within').index
    shelters_h.loc[shelters_h.index.isin(idx), 'river_risk'] = label
print(f'河流風險分布: {shelters_h["river_risk"].value_counts().to_dict()}')

## Step 4: 載入 DEM 並計算坡度
- Raster = NumPy array + GPS
- 坡度公式：`slope = arctan(sqrt(dx² + dy²))`

In [ ]:
with rasterio.open(DEM_PATH) as src:
    dem_data = src.read(1).astype(np.float64)
    dem_nodata = src.nodata
    dem_transform = src.transform
    dem_crs_rio = src.crs
    dem_profile = src.profile.copy()

if dem_nodata is not None:
    dem_data[dem_data == dem_nodata] = np.nan

pixel_size = abs(dem_transform.a)
dy, dx = np.gradient(dem_data, pixel_size)
slope_full = np.degrees(np.arctan(np.sqrt(dx**2 + dy**2)))
slope_full[np.isnan(dem_data)] = np.nan

print(f'DEM shape: {dem_data.shape}, Pixel: {pixel_size:.0f}m')
print(f'Elevation: {np.nanmin(dem_data):.0f} ~ {np.nanmax(dem_data):.0f} m')
print(f'Slope: {np.nanmin(slope_full):.1f} ~ {np.nanmax(slope_full):.1f} deg')

slope_profile = dem_profile.copy()
slope_profile.update(dtype='float64', nodata=np.nan)
slope_tif = os.path.join(tempfile.gettempdir(), 'slope_hualien.tif')
with rasterio.open(slope_tif, 'w', **slope_profile) as dst:
    dst.write(slope_full[np.newaxis, :, :])
print('Slope raster saved')

## Step 5: Zonal Statistics — 每個避難所的地形特徵
對每個避難所建立 500m 緩衝區，萃取：
- **平均高程** (mean_elevation)
- **最大坡度** (max_slope) → 用於風險分級
- **平均坡度** (mean_slope)

> **CRS 黃金法則**：Vector 對齊 Raster，不要反過來

In [ ]:
shelters_dem_crs = shelters_h.to_crs(dem_crs_rio)
shelter_buffers = shelters_dem_crs.copy()
shelter_buffers['geometry'] = shelter_buffers.geometry.buffer(500)

mean_elevs, max_slopes, mean_slopes = [], [], []

for _, row in shelter_buffers.iterrows():
    geom = [mapping(row.geometry)]
    try:
        with rasterio.open(DEM_PATH) as src:
            dem_masked, _ = rio_mask(src, geom, crop=True, nodata=-9999, filled=True)
        dem_vals = dem_masked[0].astype(np.float64)
        dem_vals[(dem_vals == -9999) | (dem_vals == dem_nodata)] = np.nan
        valid_dem = dem_vals[~np.isnan(dem_vals)]

        with rasterio.open(slope_tif) as src2:
            slope_masked, _ = rio_mask(src2, geom, crop=True, nodata=-9999, filled=True)
        slope_vals = slope_masked[0].astype(np.float64)
        slope_vals[slope_vals == -9999] = np.nan
        valid_slope = slope_vals[~np.isnan(slope_vals)]

        me = float(np.mean(valid_dem)) if len(valid_dem) > 0 else np.nan
        ms = float(np.max(valid_slope)) if len(valid_slope) > 0 else np.nan
        avg_s = float(np.mean(valid_slope)) if len(valid_slope) > 0 else np.nan
    except Exception:
        me, ms, avg_s = np.nan, np.nan, np.nan
    mean_elevs.append(me)
    max_slopes.append(ms)
    mean_slopes.append(avg_s)

shelters_h['mean_elevation'] = mean_elevs
shelters_h['max_slope'] = max_slopes
shelters_h['mean_slope'] = mean_slopes

print(f'Valid: {shelters_h["max_slope"].notna().sum()}/{len(shelters_h)}')
print(f'Elevation: {shelters_h["mean_elevation"].min():.0f}~{shelters_h["mean_elevation"].max():.0f} m')
print(f'Max slope: {shelters_h["max_slope"].min():.1f}~{shelters_h["max_slope"].max():.1f} deg')

## Step 6: 風險分級
### 地形風險（坡度）
- **LOW**: 0-15°
- **MEDIUM**: 15-30°
- **HIGH**: >30°

### 複合風險 = 河流風險 + 地形風險
| Score | Level |
|-------|-------|
| ≥5 | CRITICAL |
| 3-4 | HIGH |
| 2 | MEDIUM |
| 0-1 | LOW |

In [ ]:
def slope_risk(s):
    if pd.isna(s): return '未知'
    if s > 30: return 'HIGH'
    if s > 15: return 'MEDIUM'
    return 'LOW'

shelters_h['terrain_risk'] = shelters_h['max_slope'].apply(slope_risk)

def composite(river, terrain):
    rm = {'高風險': 3, '中風險': 2, '低風險': 1, '安全': 0}
    tm = {'HIGH': 3, 'MEDIUM': 2, 'LOW': 1, '未知': 0}
    s = rm.get(river, 0) + tm.get(terrain, 0)
    if s >= 5: return 'CRITICAL'
    if s >= 3: return 'HIGH'
    if s >= 2: return 'MEDIUM'
    return 'LOW'

shelters_h['composite_risk'] = shelters_h.apply(
    lambda r: composite(r['river_risk'], r['terrain_risk']), axis=1)

print(f'Terrain risk: {shelters_h["terrain_risk"].value_counts().to_dict()}')
print(f'Composite risk: {shelters_h["composite_risk"].value_counts().to_dict()}')
shelters_h[['避難收容處所名稱', 'TOWNNAME', 'river_risk', 'terrain_risk',
            'composite_risk', 'mean_elevation', 'max_slope']].head(10)

## Step 7: 地形風險視覺化

In [ ]:
hualien_dem_crs = hualien_boundary.to_crs(dem_crs_rio)
dem_clip = rioxarray.open_rasterio(DEM_PATH).rio.clip(hualien_dem_crs.geometry, hualien_dem_crs.crs)
nodata_v = dem_clip.rio.nodata
elev_arr = dem_clip.values[0].astype(np.float64)
if nodata_v is not None:
    elev_arr[elev_arr == nodata_v] = np.nan

dy2, dx2 = np.gradient(elev_arr, pixel_size)
slope_arr = np.degrees(np.arctan(np.sqrt(dx2**2 + dy2**2)))
slope_arr[np.isnan(elev_arr)] = np.nan

fig, axes = plt.subplots(1, 3, figsize=(20, 8))

im1 = axes[0].imshow(elev_arr, cmap='terrain', vmin=0, vmax=3500)
axes[0].set_title('Elevation (m)', fontsize=14, fontweight='bold')
plt.colorbar(im1, ax=axes[0], shrink=0.6, label='m')
axes[0].axis('off')

im2 = axes[1].imshow(slope_arr, cmap='YlOrRd', vmin=0, vmax=60)
axes[1].set_title('Slope (degrees)', fontsize=14, fontweight='bold')
plt.colorbar(im2, ax=axes[1], shrink=0.6, label='deg')
axes[1].axis('off')

risk_r = np.full_like(slope_arr, np.nan)
risk_r[slope_arr <= 15] = 1
risk_r[(slope_arr > 15) & (slope_arr <= 30)] = 2
risk_r[slope_arr > 30] = 3
risk_cmap = ListedColormap(['#2ecc71', '#f39c12', '#e74c3c'])
axes[2].imshow(risk_r, cmap=risk_cmap, vmin=1, vmax=3)
axes[2].set_title('Slope Risk', fontsize=14, fontweight='bold')
axes[2].legend(handles=[
    Patch(facecolor='#2ecc71', label='LOW (0-15 deg)'),
    Patch(facecolor='#f39c12', label='MEDIUM (15-30 deg)'),
    Patch(facecolor='#e74c3c', label='HIGH (>30 deg)')
], loc='lower right', fontsize=9)
axes[2].axis('off')

plt.suptitle(f'ARIA v2.0 - {TARGET_COUNTY} Terrain Intelligence', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('terrain_risk_map.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved terrain_risk_map.png')

## Step 8: 匯出分析結果

In [ ]:
audit = []
for _, r in shelters_h.iterrows():
    audit.append({
        'shelter_id': int(r['序號']),
        'name': r['避難收容處所名稱'],
        'town': r.get('TOWNNAME', ''),
        'river_risk': r['river_risk'],
        'terrain_risk': r['terrain_risk'],
        'composite_risk': r['composite_risk'],
        'mean_elevation_m': round(r['mean_elevation'], 1) if pd.notna(r['mean_elevation']) else None,
        'max_slope_deg': round(r['max_slope'], 1) if pd.notna(r['max_slope']) else None,
        'mean_slope_deg': round(r['mean_slope'], 1) if pd.notna(r['mean_slope']) else None,
        'capacity': int(r['預計收容人數']),
        'lon': round(float(r['經度']), 6),
        'lat': round(float(r['緯度']), 6),
    })

with open('terrain_risk_audit.json', 'w', encoding='utf-8') as f:
    json.dump(audit, f, ensure_ascii=False, indent=2)
print(f'Exported terrain_risk_audit.json ({len(audit)} records)')
print(f'\nComposite risk summary:')
print(shelters_h['composite_risk'].value_counts())

## 分析摘要

### 方法論
1. **Week 3 延伸**：OSM 河川中心線 → 500/1000/2000m 三級緩衝區 → 河流風險
2. **DEM 地形分析**：內政部 20m DEM → clip 到花蓮縣 → `np.gradient` 計算坡度
3. **Zonal Statistics**：每個避難所 500m buffer → 萃取平均高程 + 最大坡度
4. **風險分級**：坡度 0-15° LOW / 15-30° MEDIUM / >30° HIGH
5. **複合風險**：河流風險分數 + 地形風險分數 → CRITICAL / HIGH / MEDIUM / LOW

### 限制
- 河川資料為 OSM 中心線近似，非水利署官方河川區域
- DEM 僅涵蓋花蓮縣，20m 解析度
- 未納入堤防、排水設施等實際防洪因子